# Build the Neural Network

A network is built from layers. `torch.nn` provides the layers, and `nn.Module` is the base class that holds them together.

Every model is a subclass of `nn.Module` with two parts:

- `__init__` creates the layers
- `forward` says how data flows through them

Layers stored as attributes are registered automatically, so their weights show up in `model.parameters()` and move with `.to(device)`.

In [ ]:
import torch
from torch import nn

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

## Defining the model

`super().__init__()` must be called first, otherwise the module cannot register anything.

The stack takes 784 inputs, passes through two hidden layers of 512, and ends at 10 outputs, one per class.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
model = NeuralNetwork().to(device)

print(model)

## Calling the model

Call `model(x)`, never `model.forward(x)`. The call operator runs registered hooks around `forward`, and skipping it breaks anything that relies on them.

The input has to be on the same device as the weights.

In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)

print(X.shape)
print(logits.shape)
print(logits)

## Logits, probabilities, predicted class

Three different things, and mixing them up is a common bug:

- **logits** are the raw outputs of the last layer, any real number, positive or negative
- **probabilities** come from `Softmax`, they are in 0.0-1.0 and sum to 1 across the classes
- **predicted class** is `argmax`, the index of the largest value

`dim=1` because dimension 0 is the batch. Softmax has to run across the classes, not across the samples.

The model returns logits and no softmax layer is included. `nn.CrossEntropyLoss` applies its own log-softmax internally, so adding one in the model would apply it twice and hurt training.

In [ ]:
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)

print(pred_probab)
print(pred_probab.sum(dim=1).item())
print(y_pred.item())

## The layers one by one

A minibatch of 3 images, each 28x28. Watching the shape after every layer is the fastest way to understand a network.

```text
input:    [3, 28, 28]
flatten:  [3, 784]
linear:   [3, 20]
relu:     [3, 20]
linear:   [3, 10]
softmax:  [3, 10]
```

In [ ]:
input_image = torch.rand(3, 28, 28)

print(input_image.shape)

`nn.Flatten` keeps dimension 0, the batch, and flattens everything after it. 28 times 28 becomes 784. The batch dimension is never flattened away.

In [ ]:
flatten = nn.Flatten()
flat_image = flatten(input_image)

print(flat_image.shape)

`nn.Linear` applies `x @ W.T + b`. It only ever touches the last dimension, so `in_features` must match it. The weight is stored as `[out_features, in_features]`.

In [ ]:
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_image)

print(hidden1.shape)
print(layer1.weight.shape)
print(layer1.bias.shape)

`nn.ReLU` replaces every negative value with 0 and leaves the rest. The shape does not change.

Without a non-linearity between them, two stacked `Linear` layers collapse into a single `Linear` layer and the depth buys nothing.

In [ ]:
print(round(hidden1.min().item(), 4), round(hidden1.max().item(), 4))

hidden1 = nn.ReLU()(hidden1)

print(round(hidden1.min().item(), 4), round(hidden1.max().item(), 4))
print(hidden1.shape)

`nn.Sequential` is an ordered container. Data passes through the modules in the order they were given.

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10),
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

print(logits.shape)

In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

print(pred_probab.shape)
print(pred_probab.sum(dim=1))

## Parameters

Layers assigned as attributes are tracked, so their weights and biases appear in `named_parameters()` without any extra work.

`requires_grad=True` means autograd will compute a gradient for that tensor. In transfer learning some of these get frozen, which is theme 7.

In [ ]:
for name, param in model.named_parameters():
    print(name, tuple(param.shape), param.requires_grad)

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(total)
print(trainable)

The model has no training in it yet. The weights are random, so the predictions above are meaningless. Loss, gradients and the optimizer come next.